# Tomás Solano, InZenFenix, IIT414W, 22-03-2026

# 0. Reproducibility, Libraries and Cache

In [2]:
# ── Cell 1: Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings
import platform

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print("\nSystem: " + platform.system())
print("Distro: " + platform.release())

Python  : 3.14.3
NumPy   : 2.2.3
Seed    : 414

System: Linux
Distro: 6.19.9-1-cachyos


In [ ]:
# ── Cell 2: Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import fastf1                    # Formula 1 data access
from sklearn.linear_model import LogisticRegression # Model
from sklearn.metrics import ( #Scores
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.2.3


# ── Cell 2.1: Cache and validations ───────────────────────────────────────────────

required = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'seaborn', 'fastf1']
rows = []
for pkg in required:
    mod = importlib.import_module(pkg)
    rows.append((pkg, getattr(mod, '__version__', 'n/a')))

print(pd.DataFrame(rows, columns=['package', 'version']).to_string(index=False))
print(f'Working directory: {os.getcwd()}')

# FastF1 cache — auto-create if missing
cache_path = os.path.join(os.path.dirname(os.getcwd()), '../..', 'data', 'fastf1_cache')
cache_path = os.path.abspath(cache_path)
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
print(f'FastF1 cache enabled: {cache_path}')

print(subprocess.run(['git', '--version'], capture_output=True, text=True, check=False).stdout.strip())
log = subprocess.run(['git', 'log', '--oneline', '-5'], capture_output=True, text=True, check=False)
print('Recent commits:')
print(log.stdout.strip() if log.returncode == 0 else 'No commit history available in this folder.')

# 1. Feature Engineering based on F1 domain knowledge:

- The three selected features will be (plus the justification):

1. **Rolling Average Finish** (Rolling Aggregate):
    - Average finish position over the last 8 races.
    - Gets current form and reliability. If a Drivers is currently doing well, It is quite likely they'll maintain that streak.
    - Leakage Guard: We can use prior / pre-race data (e.g. shift(1)) to prevent leakage.

2. **Grid-to-Finish Reliability** (Lag Feature)
    - Delta between *GridPosition* and *ClassifiedPosition*
    - Proxy for "Race" vs. "Qualifying". Some cars might be better at qualifying than the actual race (Plus, on qualifying the Driver is alone trying to get a high score against no other Drivers), this should help balance the bias.
    - Leakage Guard: We can use one event prior to prevent leakage.

3. **Constructor Tier** (Categorical Encoding)
    - We can map the teams into 3 tiers depending on the points they got last season (Top, Mid, Bottom).
    - Regardless of the driver, if the car development and the team behind is top tier, there's a higher chance of winning.
    - Leakage Guard: Using previous years we can prevent leakage, since the points are already fixed at this point.

## Important
> There will be minor changes between the temporal validation's system on lab 1, mainly because lab 1 only used one data from one session and some races so splitting it was pretty easy for temporal validation, that system will be maintained albeit slightly modified to add data from different seasons.

# 2. Getting the Data

In [ ]:
# Requirements
RANDOM_SEED = 414

# 1. Define Constructor Tiers based on 2023 Season Results (Pre-race knowledge)
# This serves as our Categorical/Ordinal Encoding feature
tier_map = {
    'Red Bull Racing': 3, 'Mercedes': 3, 'Ferrari': 3, 'McLaren': 3,
    'Aston Martin': 2, 'Alpine': 2, 'Williams': 2,
    'Haas F1 Team': 1, 'RB': 1, 'Kick Sauber': 1
}

# 2. Collect Data chronologically
all_race_data = []
race_nums = range(1, 24) # Expanded range to ensure we have "history" for rolling averages

for race_num in race_nums:
    try:
        session = fastf1.get_session(2024, race_num, 'R')
        session.load(laps=False, telemetry=False)
        
        # We need Grid for our model and Position for our 'Form' calculation
        res = session.results[['FullName', 'TeamName', 'GridPosition', 'ClassifiedPosition']]
        res['Round'] = race_num
        all_race_data.append(res)
    except:
        continue

full_df = pd.concat(all_race_data).reset_index(drop=True)

# 3. Feature Engineering (The 3 New Features)

# Helper to clean position strings (converts 'R', 'W', etc. to 20 for math purposes)
def clean_pos(x):
    return int(x) if str(x).isdigit() else 20

full_df['PosNumeric'] = full_df['ClassifiedPosition'].apply(clean_pos)

# FEATURE 1: Rolling Average (Rolling Aggregate)
# Average finish of the last 3 races. We SHIFT(1) to prevent leakage.
full_df['rolling_form'] = full_df.groupby('FullName')['PosNumeric'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# FEATURE 2: Grid Position (Direct Pre-race Feature)
# This is naturally pre-race, so no shift needed.
full_df['grid_num'] = full_df['GridPosition'].astype(int)

# FEATURE 3: Team Tier (Categorical/Ordinal Encoding)
full_df['team_tier'] = full_df['TeamName'].map(tier_map).fillna(1)

# Target Variable
full_df['IsTop10'] = full_df['PosNumeric'].apply(lambda x: 1 if x <= 10 else 0)

# 4. Final Cleanup & Temporal Split
# Drop rows where we don't have rolling history (the first race for each driver)
data_df = full_df.dropna(subset=['rolling_form']).sort_values('Round')

# Selecting our 3 Engineered Features + Target
features = ['grid_num', 'rolling_form', 'team_tier']
X = data_df[features]
y = data_df['IsTop10']

# Maintain your 67% Temporal Split
split_index = int(len(data_df) * 0.67)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Features used: {features}")
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

display(X)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
/tmp/ipykernel_89643/3648257072.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res['Round'] = race_num
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_c

Features used: ['grid_num', 'rolling_form', 'team_tier']
Training samples: 292, Testing samples: 144


/tmp/ipykernel_89643/3648257072.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res['Round'] = race_num


,grid_num,rolling_form,team_tier
20,1,1.000000,3
21,3,2.000000,3
22,2,4.000000,3
23,5,8.000000,3
31,13,12.000000,1
...,...,...,...
445,8,15.000000,2
442,1,3.333333,3
439,2,4.000000,3
440,5,4.000000,3


# 3. Logistic Regression Model

Since we are trying to get a binary result (either top 10 or not), we can use a logistic regression to get that result



In [14]:
model = LogisticRegression(random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

#We get the scores
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_pred)


print("------------ SCORES LOGISTIC REGRESSION MODEL------------")

print(f"Accuracy: {acc:.2f}")
print(f"Precision: {prec:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1_Score: {f1:.2f}")
print(f"ROC-AUC: {roc:.2f}")

------------ SCORES LOGISTIC REGRESSION MODEL------------
Accuracy: 0.78
Precision: 0.79
Recall: 0.74
F1_Score: 0.76
ROC-AUC: 0.78


## 4. Miscellanous Data (Confusion Matrix)

> Added Data for better understanding of the obtained results

In [18]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(cm, columns = ["Actual Positive", "Actual Negative"], index=["Predicted Positive", "Predicted Negative"])
display(cm_df)

,Actual Positive,Actual Negative
Predicted Positive,60,14
Predicted Negative,18,52


## 5. Error analysis

> Errors on the model:

- False Positives: 18
- False Negatives: 14

In [19]:
error_analysis_df = full_df.loc[X_test.index].copy()
error_analysis_df['Predicted'] = y_pred
error_analysis_df['Actual'] = y_test.values

# 3. Identify False Positives (Model said Top 10, but they failed)
false_positives = error_analysis_df[(error_analysis_df['Predicted'] == 1) & (error_analysis_df['Actual'] == 0)]

# 4. Identify False Negatives (Model said Bottom 10, but they got Top 10)
false_negatives = error_analysis_df[(error_analysis_df['Predicted'] == 0) & (error_analysis_df['Actual'] == 1)]

print("--- TOP FALSE POSITIVES (Predicted Top 10, but didn't make it) ---")
print(false_positives[['Round', 'FullName', 'GridPosition', 'PosNumeric']].head(5))

print("\n--- TOP FALSE NEGATIVES (Predicted Bottom 10, but made Top 10) ---")
print(false_negatives[['Round', 'FullName', 'GridPosition', 'PosNumeric']].head(5))

--- TOP FALSE POSITIVES (Predicted Top 10, but didn't make it) ---
     Round         FullName  GridPosition  PosNumeric
335     17     Sergio Perez           4.0          17
336     17     Carlos Sainz           3.0          18
369     19  Kevin Magnussen           8.0          11
370     19     Pierre Gasly           6.0          12
371     19  Fernando Alonso           7.0          13

--- TOP FALSE NEGATIVES (Predicted Bottom 10, but made Top 10) ---
     Round         FullName  GridPosition  PosNumeric
308     16  Kevin Magnussen          13.0          10
322     17     Lando Norris          15.0           4
327     17   Lewis Hamilton          19.0           9
328     17   Oliver Bearman          10.0          10
348     18     Sergio Perez          13.0          10


## Analysis:

- 1. The main takeaway is that the model heavily relies on grid position to predict if someone is top 10, meaning that If a Driver doesn't perform well and ends up losing its grid advantage, It is much more likely to get the prediction wrong.

- 2. Another problem is that the model doesn't give that much importance to how a driver performs, someone like Norris, which is known to be a top Driver, went from 15 to 4 no problem, and viceversa, some Drivers that didn't perform as well but got a good grid position ended up outside the top 10, which could be because of weather, the car or the Driver themselves.

- 3. The last big point is that 3 features don't tell the whole story, just as it relies heavily on grid position, it is also important to get data from the driver's and team themselves, here we did a bit of that adding domain knowledge (team tiers, average position) but if more of this features are added we could balance out the grid position with other factors.

- 4. There's also always the problem of missing values like DNF or what not, where a Driver can get disqualified or the whole car dies in the middle of a race, this is one point where this model is heavily faulty since no data from the cars itself that the Drivers uses and the teams creates/maintains were used.